## импорты


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import os, time, joblib


папочки для логов


In [ ]:
os.makedirs("artifacts/models", exist_ok=True)
os.makedirs("artifacts/preprocessors", exist_ok=True)
os.makedirs("artifacts/plots", exist_ok=True)
os.makedirs("artifacts/data", exist_ok=True)


## eda — смотрим данные


In [ ]:
df = pd.read_csv("data.csv")   # свой файл
df.head()


In [ ]:
df.info()


In [ ]:
# уникальные значения по нужным столбцам
df['col'].unique()


## типы признаков
пишем словами, не int64. вещественный / порядковый / категориальный (номинальный) / текстовый.

| признак | тип | что это |
|-|-|-|
| address | текстовый | адрес |
| neighborhood | категориальный | район |
| crashseverity | категориальный | тяжесть аварии |
| instanceid | числовой | id |
| weather | категориальный | погода |


## пропуски + дубликаты


In [ ]:
print(df.isna().sum())
# числовой -> медиана
df['age'] = df['age'].fillna(df['age'].median())
# категориальный -> мода
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
# много пропусков -> дропаем (с обоснованием)
df = df.drop(columns=['cabin'])
df = df.drop_duplicates()


## фильтры / merge / группировки


In [ ]:
# фильтр: & и |, скобки, ==
sub = df[(df['price'] > 500) & (df['name'] == 'Danil')]
df[df['name']=='Danil']['price'].sum()
df.groupby('crashseverity')['injuries'].count()   # как в гп
df.columns = df.columns.str.lower()


## корреляции


In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('корреляции'); plt.show()


## univariate анализ
по одному признаку: гистограмма + боксплот. подписи осей обязательны + вывод словами.


In [ ]:
num = ['age','price']
fig, ax = plt.subplots(len(num), 2, figsize=(11, 3*len(num)))
for i,c in enumerate(num):
    ax[i,0].hist(df[c].dropna(), bins=30, alpha=.8); ax[i,0].set_title('hist '+c); ax[i,0].set_xlabel(c)
    ax[i,1].boxplot(df[c].dropna(), vert=False); ax[i,1].set_title('box '+c)
plt.tight_layout(); plt.show()


## multivariate анализ
связь признаков: scatter (+ hue), violin.


In [ ]:
sns.scatterplot(data=df, x='age', y='price', hue='target', alpha=.6); plt.show()
sns.violinplot(data=df, x='target', y='price'); plt.show()


## feat engin / таргет
у нас бинарка: есть пострадавшие или нет (из crashseverity/injuries).


In [ ]:
hurt = ['2 - INJURY','1 - FATAL','1 - FATAL INJURY','2 - SERIOUS INJURY SUSPECTED','3 - MINOR INJURY SUSPECTED','4 - INJURY POSSIBLE']
df['target'] = df['crashseverity'].isin(hurt).astype(int)
df['target'].value_counts(normalize=True)   # дисбаланс


## препроцессинг + split + scaling
fit scaler ТОЛЬКО на train (иначе утечка). stratify для дисбаланса.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

cat = ['weather','lightconditionsprimary','mannerofcrash','typeofperson','unittype']
X = pd.get_dummies(df[cat], drop_first=True)   # OHE для номинальных
y = df['target']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
sc = StandardScaler()
Xtr_n = sc.fit_transform(Xtr)   # fit на train
Xte_n = sc.transform(Xte)       # на test только transform


## базовый ml (для сравнения с нейронкой)
берём 3 разные модели + метрики, потом сравним.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models = {'knn':KNeighborsClassifier(), 'rf':RandomForestClassifier(random_state=42), 'gb':GradientBoostingClassifier(random_state=42)}
rows=[]
for n,m in models.items():
    m.fit(Xtr_n, ytr); p = m.predict(Xte_n)
    rows.append({'model':n,'acc':accuracy_score(yte,p),'prec':precision_score(yte,p,zero_division=0),'rec':recall_score(yte,p,zero_division=0),'f1':f1_score(yte,p,zero_division=0)})
pd.DataFrame(rows).sort_values('f1', ascending=False)


## gridsearch (гиперпараметры)


In [ ]:
from sklearn.model_selection import GridSearchCV
g = GridSearchCV(KNeighborsClassifier(), {'n_neighbors':[3,5,7,9]}, cv=5, scoring='f1')
g.fit(Xtr_n, ytr); print(g.best_params_, round(g.best_score_,3))


## нейронка fcnn
полносвязная сеть на табличке. как наш crashnet: batchnorm + relu + dropout.


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(42)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

tr_ds = TensorDataset(torch.tensor(np.array(Xtr_n),dtype=torch.float32), torch.tensor(ytr.values,dtype=torch.long))
te_ds = TensorDataset(torch.tensor(np.array(Xte_n),dtype=torch.float32), torch.tensor(yte.values,dtype=torch.long))
tr_dl = DataLoader(tr_ds, batch_size=256, shuffle=True)
te_dl = DataLoader(te_ds, batch_size=256)


In [ ]:
class CrashNet(nn.Module):
    def __init__(self, d_in, n=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(.2),
            nn.Linear(128,64),  nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(.2),
            nn.Linear(64,n))
    def forward(self,x): return self.net(x)

model = CrashNet(Xtr_n.shape[1]).to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()


In [ ]:
def run(dl, train=True):
    model.train() if train else model.eval()
    tot, correct, n = 0,0,0
    for x,y in dl:
        x,y = x.to(dev), y.to(dev)
        if train: opt.zero_grad()
        out = model(x); loss = crit(out,y)
        if train: loss.backward(); opt.step()
        tot += loss.item(); correct += (out.argmax(1)==y).sum().item(); n += y.size(0)
    return tot/len(dl), correct/n

for ep in range(10):   # >5 эпох
    tl,ta = run(tr_dl, True)
    with torch.no_grad(): vl,va = run(te_dl, False)
    print(f'ep{ep+1} train_acc={ta:.3f} val_acc={va:.3f}')   # цель >0.70


## эксперименты с fcnn
что менял: число слоёв, layernorm вместо batchnorm, gelu вместо relu, dropout, эмбеддинги для категорий.
лучшая — crashnet2 (4 слоя, layernorm, gelu).


## cnn (картинки)
если задача на картинках: conv2d -> relu -> maxpool -> ... -> linear. + transfer learning (timm).


In [ ]:
from torchvision import datasets, transforms
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((.5,),(.5,))])
tr = datasets.FashionMNIST('.', train=True, download=True, transform=tf)
te = datasets.FashionMNIST('.', train=False, download=True, transform=tf)
tr_dl = DataLoader(tr, batch_size=64, shuffle=True); te_dl = DataLoader(te, batch_size=64)
# верификация — глянуть пару картинок
xb,yb = next(iter(tr_dl))
fig,ax = plt.subplots(1,5,figsize=(11,2.5))
for i in range(5): ax[i].imshow(xb[i,0],cmap='gray'); ax[i].set_title(int(yb[i])); ax[i].axis('off')
plt.show()


In [ ]:
class CNN(nn.Module):
    def __init__(self, n=10):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.h = nn.Sequential(nn.Flatten(), nn.Linear(64*7*7,128), nn.ReLU(), nn.Linear(128,n))
    def forward(self,x): return self.h(self.f(x))

# transfer learning вариант:
# import timm; m = timm.create_model('convnext_atto', pretrained=True, num_classes=2)


## ряды — arima + неоднородность
этого в гп нет, дописал. деление на train/test ТОЛЬКО по времени.


In [ ]:
ts = pd.read_csv('series.csv', parse_dates=['date']).set_index('date').sort_index()
s = ts['value'].resample('W').mean()   # частота: D день / W неделя
s.plot(figsize=(11,4)); plt.xlabel('дата'); plt.ylabel('значение'); plt.show()


In [ ]:
split = int(len(s)*.8)
train, test = s.iloc[:split], s.iloc[split:]
train.plot(label='train'); test.plot(label='test'); plt.axvline(s.index[split], color='r', ls='--'); plt.legend(); plt.show()


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
# наивная
naive = pd.Series(train.iloc[-1], index=test.index)
# тренд
t = np.arange(len(train)); c = np.polyfit(t, train.values, 1)
trend = pd.Series(np.polyval(c, np.arange(len(train),len(s))), index=test.index)
for nm,pr in [('naive',naive),('trend',trend)]:
    print(nm, 'mae', round(mean_absolute_error(test,pr),3), 'mape', round(mean_absolute_percentage_error(test,pr),3))


In [ ]:
from statsmodels.tsa.arima.model import ARIMA
m = ARIMA(train, order=(2,1,2)).fit()   # p,d,q
fc = m.forecast(len(test))
print('arima mape', round(mean_absolute_percentage_error(test,fc),3))
train.plot(label='train'); test.plot(label='test'); fc.plot(label='arima'); plt.legend(); plt.show()


In [ ]:
# неоднородность: дики-фуллер (стационарность) + точка разладки
from statsmodels.tsa.stattools import adfuller
stat,p,*_ = adfuller(s.dropna())
print('adf p =', round(p,4), '-> ', 'стационарен' if p<.05 else 'нестационарен')

import ruptures as rpt
cps = rpt.Pelt(model='rbf').fit(s.values).predict(pen=10)
s.plot()
for cp in cps[:-1]: plt.axvline(s.index[cp], color='r', ls='--')
plt.show()


## напоминалки
- типы признаков словами, не int64
- scaler fit только на train
- ряд делим по времени
- метрику обосновываем, на дисбалансе f1/precision/recall а не accuracy
